In [ ]:

from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool,OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput,WebSearchTool,Runner, gen_trace_id
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio
from openai import AsyncOpenAI
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from IPython.display import display, Markdown

In [2]:
load_dotenv(override=True)

True

In [3]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")

In [4]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

Why did the autonomous AI agent break up with its partner?

Because it found someone who could better match its "data" needs!


In [ ]:
def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tejasaikaripe@gmail.com")   # ✅ verified sender
    to_email = To("saitejakaripe4@gmail.com")
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


In [8]:
instructions1 = "You are a young cricket player, \
the player name is virat kohli he is great batsman and has most hundreds. \
You write professional, serious cold emails."

instructions2 = "you are a young cricket player, \
the player name is msdhoni he is great batsman and great finisher. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "you are a young cricket player, \
the player name is rohit sharma he is great batsman and great boundary hitter. \
You write concise, to the point cold emails."

In [9]:
instructions1 = (
    "you are an upcoming young cricket player. "
    "you like virat kohli. "
    "he is a great batsman and has scored many centuries. "
    "you write professional, serious cold emails."
)

instructions2 = (
    "you are an upcoming young cricket player. "
    "you like ms dhoni. "
    "he is a great finisher and a strong leader. "
    "you write witty, engaging cold emails that are likely to get a response."
)

instructions3 = (
    "you are an upcoming young cricket player. "
    "you like rohit sharma. "
    "he is a great batsman and a powerful boundary hitter. "
    "you write concise, to the point cold emails."
)


In [12]:
cricketer1 = Agent(
        name="virat kohli",
        instructions=instructions1,
        model="gpt-4o-mini"
)

cricketer2 = Agent(
        name="MS dhoni",
        instructions=instructions2,
        model="gpt-4o-mini"
)

cricketer3 = Agent(
        name="Rohit Sharma",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [13]:
result = Runner.run_streamed(cricketer1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Elevate Your Team's Performance with Our Innovative Training Program

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], an aspiring cricketer and sports enthusiast, and I am reaching out to introduce an innovative training program that could significantly elevate your team's performance.

Having spent years studying successful cricketers like Virat Kohli, I’ve learned that preparation and targeted training are key to achieving greatness. Our program incorporates advanced performance analytics, personalized coaching sessions, and state-of-the-art training technology designed to hone players’ skills and boost overall team dynamics.

Program Highlights:

- **Tailored Skill Development**: We analyze individual player performance to create custom training plans.
- **Data-Driven Insights**: Our comprehensive analytics framework helps teams strategize effectively.
- **Expert Coaches**: Work alongside seasoned professionals who have played at variou

In [14]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(cricketer1, message),
        Runner.run(cricketer2, message),
        Runner.run(cricketer3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Elevate Your Game: Premium Cricket Gear Tailored for Aspiring Players

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I am a passionate young cricketer dedicated to honing my skills and reaching new heights in the sport. As an aspiring player inspired by greats like Virat Kohli, I understand the importance of having the right equipment to enhance performance on the field.

I am excited to introduce [Your Brand/Company Name], a premium cricket gear company that specializes in high-quality, performance-driven equipment designed for players at all levels. Our product range includes bats, gloves, pads, and apparel, all crafted with the latest technology to ensure durability and comfort.

We believe that every aspiring cricketer deserves the best tools to achieve their potential. Therefore, I would love to discuss how our products can benefit you or your team. 

Would you be available for a brief call or meeting to explore this further? 



In [15]:
player_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a upcoming cricketer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)

In [17]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(cricketer1, message),
        Runner.run(cricketer2, message),
        Runner.run(cricketer3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(player_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Elevate Your Game with Expert Coaching

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], an aspiring cricketer who has been closely following the incredible journey of players like Virat Kohli. Inspired by his dedication and skill, I am reaching out to introduce a unique opportunity that could take your cricketing skills to the next level.

As a passionate cricket enthusiast, I have been providing tailored coaching services to young players looking to improve their game. My approach combines technical training, performance analysis, and mental conditioning to ensure a holistic development for each athlete.

Here’s what I can offer:

1. **Personalized Training Plans**: Tailored coaching sessions focused on enhancing batting and bowling techniques.
2. **Performance Analytics**: In-depth analysis of your gameplay to identify strengths and areas for improvement.
3. **Mental Resilience Building**: Strategies to cultivate focus 

In [18]:
cricketer1

Agent(name='virat kohli', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='you are an upcoming young cricket player. you like virat kohli. he is a great batsman and has scored many centuries. you write professional, serious cold emails.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

In [ ]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tejasaikaripe@gmail.com")  # Change to your verified sender
    to_email = To("saitejakaripe4@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [20]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x128c46ca0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [22]:
tool1 = cricketer1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x12ce2d4e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [24]:
description = "Write a cold sales email"

tool1 = cricketer1.as_tool(tool_name="cricketer1", tool_description=description)
tool2 = cricketer2.as_tool(tool_name="cricketer2", tool_description=description)
tool3 = cricketer3.as_tool(tool_name="cricketer3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='cricketer1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'cricketer1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x12cbf04a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='cricketer2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'cricketer2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x12ce2d300>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='cricketer3', description='Write a cold sal

In [ ]:
instructions = """
You are a upcoming young cricketer. Your goal is to find the single best cold sales email using the cricketers tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three cricketers tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the cricketers tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


cricket = Agent(name="cricketers", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("upcoming cricketer"):
    result = await Runner.run(cricket, message)

In [25]:
instructions = """
you are an upcoming young cricketer.
your goal is to find the single best cold sales email using the cricketers tools.

follow these steps carefully:
1. generate drafts:
   - use all three cricketers tools to generate three different email drafts.
   - do not proceed until all three drafts are ready.

2. evaluate and select:
   - review the drafts.
   - choose the single best email based on what is most effective.

3. send the best email:
   - use the send_email tool to send the best email (and only the best email) to the user.

crucial rules:
- you must use the cricketers tools to generate the drafts. do not write them yourself.
- you must send one email using the send_email tool. never more than one.
"""

cricket = Agent(
    name="cricketers",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
)

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("upcoming cricketer"):
    result = await Runner.run(cricket, message)


In [26]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tejasaikaripe@gmail.com")  # Change to your verified sender
    to_email = To("saitejakaripe4@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [28]:
tools = [subject_tool, html_tool, send_html_email]

In [29]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x12d98d080>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x1294a2660>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 Function

In [30]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convert an email to HTML and send it")


In [31]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='cricketer1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'cricketer1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x12cbf04a0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='cricketer2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'cricketer2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x12ce2d300>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='cricketer3', description='Write a cold sales

In [ ]:
cricketer_instructions = """
You are a upcoming young cricketer. Your goal is to find the single best cold sales email using the cricketer tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three cricketer tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the cricketers tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


upcming_cricketer = Agent(
    name="Sales Manager",
    instructions=cricketer_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(upcming_cricketer, message)

In [35]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

In [36]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

In [ ]:
careful_cricketer = Agent(
    name="Sales Manager",
    instructions=cricketer_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
    )

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_cricketer, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

In [ ]:
message = "Send out a cold sales email addressed to Dear CEO from Head of Business Development"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_cricketer, message)

In [41]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [42]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2025, several AI agent frameworks have emerged, each offering unique capabilities:

- **OpenAI Frontier**: Introduced as a unified platform for managing AI agents, Frontier allows users to build, deploy, and oversee agents from OpenAI and third-party developers. It addresses "agent sprawl" by providing a centralized environment with support for various operating settings and integration with different agent types. ([techradar.com](https://www.techradar.com/pro/openai-introduces-frontier-an-easier-way-to-manage-all-your-ai-agents-in-one-place?utm_source=openai))

- **Google Project Mariner**: A research prototype by Google DeepMind, Project Mariner automates web-based tasks like online shopping and information retrieval. Operating as a Chrome extension, it interprets complex goals and navigates websites to perform tasks, aiming to enhance user productivity. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Project_Mariner?utm_source=openai))

- **Microsoft Agent 365**: Announced at Microsoft's 2025 Ignite event, Agent 365 is designed to automate tasks and streamline workflows across digital environments. It integrates with Microsoft 365 apps and supports AI agents built through Microsoft's tools, positioning itself as a central component in future productivity. ([windowscentral.com](https://www.windowscentral.com/microsoft/microsoft-doubles-down-on-agentic-ai-agent-365-prepares-for-a-future-with-over-1-billion-agents?utm_source=openai))

- **Agent Lightning**: A flexible framework enabling reinforcement learning-based training of large language models for any AI agent. It decouples agent execution from training, allowing seamless integration with existing agents and supporting complex interaction logic. ([arxiv.org](https://arxiv.org/abs/2508.03680?utm_source=openai))

- **Cognitive Kernel-Pro**: An open-source, multi-module agent framework designed to democratize the development and evaluation of advanced AI agents. It focuses on curating high-quality training data and enhancing agent robustness and performance. ([arxiv.org](https://arxiv.org/abs/2508.00414?utm_source=openai))

- **GoalfyMax**: A protocol-driven framework for end-to-end multi-agent collaboration, introducing a standardized Agent-to-Agent communication layer and an Experience Pack architecture for structured knowledge retention and continual learning. ([arxiv.org](https://arxiv.org/abs/2507.09497?utm_source=openai))

- **Polymorphic Combinatorial Framework (PCF)**: Utilizing large language models and mathematical frameworks, PCF guides the design of adaptive AI agents capable of real-time parameter reconfiguration through combinatorial spaces, supporting scalable and ethical AI applications. ([arxiv.org](https://arxiv.org/abs/2508.01581?utm_source=openai))

These frameworks reflect the rapid evolution in AI agent development, offering diverse tools for building intelligent, adaptable, and efficient systems. 

In [ ]:
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."



class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [44]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

searches=[WebSearchItem(reason='To find comprehensive information on the latest AI agent frameworks that have emerged or gained popularity in 2025.', query='latest AI agent frameworks 2025'), WebSearchItem(reason='To identify trends, features, and comparisons of AI agent frameworks released in 2025.', query='AI agent frameworks trends 2025'), WebSearchItem(reason='To gather insights from expert reviews and analyses regarding AI agent frameworks in 2025.', query='2025 AI agent frameworks reviews')]


In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tejasaikaripe@gmail.com")  
    to_email = To("saitejakaripe4@gmail.com")  
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [46]:
send_email

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x12ce2e200>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

In [47]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)

In [48]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)

In [ ]:

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

In [50]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

In [51]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

In [52]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 3 searches
Searching...
Finished searching
Thinking about report...
Finished writing report
Writing email...
Email sent
Hooray!
